# w2 · LSTM for churn (PyTorch)

A single LSTM model predicts churn from the **90-day × 20-channel** tensors built in `w2_sequence_preprocessing.ipynb`.

The primary KPI is **log-loss**, because it measures the quality of the predicted probabilities. The evaluation is temporal:

| Part | Cohort | Use |
|---|---|---|
| `fit` | February | training |
| `stop` | February | validation and early stopping |
| `tune` | February | Optuna selection and threshold |
| `test` | March | final evaluation, once |
| `development` | All February | final refit |

This is a single model, with no global model state and no multi-model comparison. The notebook only orchestrates the pieces that live in `src/models/lstm/` and `src/models/evaluation/`.

## 1. Configuration

The model settings, paths and Optuna budget live in `src/models/lstm/config.py`. Here they are imported and displayed so it is explicit what the run uses.

- `N_TRIALS=5`: five candidate configurations.
- `EPOCHS=40`: maximum epochs per trial.
- `PATIENCE=5`: early stopping when `stop` does not improve for five epochs.
- `BATCH_SIZE=512`: block/batch size.
- `layer_dim` is searched between 2 and 4; `hidden_dim` among 32, 64 and 128.
- `lr` and `weight_decay` are searched on a logarithmic scale.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

warnings.filterwarnings("ignore")

from src.models.lstm.config import (
    ARTIFACTS_DIR,
    BATCH_SIZE,
    CHECKPOINT_DIR,
    EPOCHS,
    EVALUATION_DIR,
    EXPORT_DIR,
    FEATURE_DIR,
    HIDDEN_DIMS,
    INPUT_DIM,
    LAYER_DIM_RANGE,
    LR_RANGE,
    MODEL_ID,
    N_TRIALS,
    OPTUNA_DIR,
    OUTPUT_DIM,
    PATIENCE,
    SEED,
    WEIGHT_DECAY_RANGE,
)
from src.models.lstm.train import resolve_device

DEVICE = resolve_device()
settings = pd.DataFrame(
    [
        ("model_id", MODEL_ID),
        ("export_dir", str(EXPORT_DIR)),
        ("feature_dir", str(FEATURE_DIR)),
        ("artifacts_dir", str(ARTIFACTS_DIR)),
        ("evaluation_dir", str(EVALUATION_DIR)),
        ("optuna_dir", str(OPTUNA_DIR)),
        ("checkpoint_dir", str(CHECKPOINT_DIR)),
        ("device", str(DEVICE)),
        ("seed", SEED),
        ("n_trials", N_TRIALS),
        ("epochs", EPOCHS),
        ("patience", PATIENCE),
        ("batch_size", BATCH_SIZE),
        ("input_dim", INPUT_DIM),
        ("output_dim", OUTPUT_DIM),
        ("hidden_dims", HIDDEN_DIMS),
        ("layer_dim_range", LAYER_DIM_RANGE),
        ("lr_range", LR_RANGE),
        ("weight_decay_range", WEIGHT_DECAY_RANGE),
    ],
    columns=["setting", "value"],
)
settings

## 2. Feature preparation

### 2.1 Data

`export-sequential-v5` contains a `(users, 90, 20)` `float16` tensor, with days ordered from oldest to newest. The channels are:

- `0:8`: listening. Seven counters transformed with `log1p` plus the `active` flag.
- `8:13`: daily transaction events.
- `13:20`: subscription state propagated forward.

The `is_churn` labels are aligned row by row with the tensor. Everything is computed from information available up to each cohort cutoff, so there is no temporal leakage.

The tensors are read with `mmap_mode="r"`; they are never fully loaded into RAM.

In [ ]:
from src.models.lstm.data import load_prepared_data

data = load_prepared_data(feature_dir=FEATURE_DIR, export_dir=EXPORT_DIR)

print("tensor:", data.view.shape, data.view.dtype)
print("input_dim:", data.input_dim)
print("channels:", data.meta["sequences"]["lstm"]["channels"])
print("split sizes:", {part: len(rows) for part, rows in data.split.items()})
for part in ("fit", "stop", "tune", "test"):
    labels = data.labels(part)
    print(f"{part:4s} n={len(labels):>9,} | churn_rate={labels.mean():.4f}")

### 2.2 Split

February is split once with a stable `msno` hash, stratified by label:

- `fit`: 75%.
- `stop`: 12.5%.
- `tune`: 12.5%.
- `test`: all of March, never used for any decision.
- `development`: all of February; used only for the final refit before predicting March.

This keeps the threshold honest: it is selected on `tune` and never re-searched on `test`.

In [ ]:
split_table = pd.DataFrame(
    [
        {
            "part": part,
            "n_users": len(data.split[part]),
            "churn_rate": float(data.labels(part).mean()),
        }
        for part in ("fit", "stop", "tune", "test", "development")
    ]
)
split_table

### 2.3 Prepared dataset and batches

The existing `PreparedSequenceData` class encapsulates:

- the labels,
- the split indices,
- the memory-mapped tensor view,
- the scalers fitted per phase.

Its `iter_batches` method returns blocks of transformed rows as PyTorch tensors. It does not transform row by row: it reads blocks to reduce overhead and keep memory usage bounded.

- `phase="fit"`: scaler fitted only on `fit`.
- `phase="development"`: scaler fitted on all of February.
- `drop_last=True`: avoids a final batch of size 1, which `BatchNorm1d` cannot normalise during training.

In [ ]:
sample_rows = data.rows("fit")[:BATCH_SIZE]
xb, yb = next(
    iter(data.iter_batches(sample_rows, "fit", BATCH_SIZE, drop_last=True))
)
print("x:", xb.shape, xb.dtype)
print("y:", yb.shape, yb.dtype)
print("first user, channel 0, day 0:", float(xb[0, 0, 0]))

## 3. Model and training

### 3.1 LSTM architecture

The input is `(batch, 90, 20)`. The network stacks between **2 and 4 LSTM layers**. Each layer uses the `tanh` nonlinearity internally: the hidden state stays bounded and training is more stable. The final hidden output `h_T` summarises the 90-day history.

After `h_T`:

1. `BatchNorm1d(hidden_dim)` stabilises the hidden-state distribution.
2. `Linear(hidden_dim, 1)` produces a logit.
3. At prediction time, `Sigmoid` turns the logit into a churn probability.

There is no dropout, as specified. Overfitting is controlled by `weight_decay`, BatchNorm and early stopping.

The loss is `BCEWithLogitsLoss`: the numerically stable version of binary cross-entropy that works directly with logits.

In [ ]:
from src.models.lstm.model import LSTMModel

example_model = LSTMModel(
    input_dim=INPUT_DIM,
    hidden_dim=64,
    layer_dim=4,
    output_dim=OUTPUT_DIM,
).to(DEVICE)

logits, hn, cn = example_model(xb.to(DEVICE))
print("logits:", tuple(logits.shape))
print("hn:", tuple(hn.shape))
print("cn:", tuple(cn.shape))
print("example probabilities:", torch.sigmoid(logits[:3]).detach().cpu().numpy().ravel())

### 3.2 Optimisation

- **AdamW** decouples weight decay from the adaptive step.
- `lr` and `weight_decay` are searched with Optuna.
- **Gradient clipping** at norm 1.0 for LSTM stability.
- **AMP** on GPU for speed.
- At most **40 epochs** per trial.
- **Early stopping** with `patience=5`: if `stop` does not improve, training stops and the best checkpoint is restored.
- No dropout.

### 3.3 Optuna

Optuna uses TPE and minimises `log_loss` on `tune`:

- trains on `fit`,
- validates on `stop`,
- evaluates on `tune`,
- restores the best `stop` state before computing the objective.

The search space is:

- `layer_dim`: 2–4.
- `hidden_dim`: 32, 64, 128.
- `lr`: `1e-4` to `1e-2` on a logarithmic scale.
- `weight_decay`: `1e-6` to `1e-2` on a logarithmic scale.
- `batch_size`: fixed at 512.

Artifacts are written under `OPTUNA_DIR`: the SQLite study, the trials CSV and the best-parameters JSON.

In [ ]:
from src.models.lstm.tune import tune_model

best = tune_model(
    data,
    n_trials=N_TRIALS,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    verbose=True,
)
print("best params:", best["best_params"])
print("best epoch:", best["best_epoch"])
print("stopped epoch:", best["stopped_epoch"])
print("tune log-loss:", best["best_value"])
print("threshold:", best["threshold"])

## 4. Test

The threshold was already selected on `tune` and is now frozen. The final refit uses all of February (`development`) for the `best_epoch` found during tuning. March is then predicted once.

A checkpoint with the architecture, parameters, threshold and epoch count is also saved.

In [ ]:
from src.models.lstm.test import test_model

results = test_model(
    best["best_params"],
    data,
    threshold=best["threshold"],
    best_epoch=best["best_epoch"],
    device=DEVICE,
    verbose=True,
)
print("test probabilities:", results["proba_test"].shape)
print("predicted churn rate:", results["pred_test"].mean())
print("threshold:", results["threshold"])
print("epochs:", results["epochs"])
print("checkpoint:", results["checkpoint"])

## 5. Results

`evaluate_model` produces the complete evaluation for the single model:

- `test` metrics at the `tune` threshold,
- complementary metrics at `0.5`,
- classification report,
- confusion matrix,
- ROC/PR curves,
- reliability/histogram,
- threshold manifest,
- Optuna history.

`log_loss` measures probability quality. `ROC-AUC` and `average_precision` measure ranking. `F1`, `precision` and `recall` measure decisions at the chosen threshold.

In the confusion matrix:

- **FN**: churners the model missed.
- **FP**: loyal customers flagged as at risk.

The threshold was selected in February and is not optimised again in March.

In [ ]:
from src.models.evaluation import evaluate_model

report = evaluate_model(
    results,
    best,
    data,
    out_dir=EVALUATION_DIR,
    show_plots=False,
)
report["display"]

In [ ]:
for key in ("confusion_png", "roc_pr", "calibration", "optuna_history"):
    path = report["paths"].get(key)
    if path is not None and Path(path).is_file():
        print(key, "->", path)
        display(Image(filename=str(path)))

## 6. Artifacts and limitations

Generated artifacts:

```text
artifacts/sherlock_w2/
├── feature_preparation/       # split, labels and scalers
└── lstm/
    ├── optuna/                # study, trials and best params
    ├── checkpoints/lstm.pt    # final model
    └── evaluation/
        ├── metrics/
        ├── figures/
        └── selection_manifest.json
```

Limitations:

- There is only one test month: March.
- Validation is used for early stopping, tuning and the threshold, so validation metrics are optimistic.
- No probability calibration was applied.
- This is a single model, not a comparison across families.
- Churn prevalence changes between February and March; that shift is part of the honest evaluation.